# ApexPlanet Data Analytics Internship
## Task 1: Data Immersion & Wrangling
**Student:** Venkata Vedhadri Marisetti (23981A44Y9)  
**Degree:** B.Tech CSE (Data Science), Raghu Engineering College (Autonomous)  
**Guide:** Md Imam Khader Shariff, Assistant Professor  
**Organization:** ApexPlanet Software Pvt. Ltd. | Certificate ID: APSPL2650541  

---
### Objectives:
1. Understand dataset structure and schema.
2. Detect missing, duplicate, and inconsistent data entries.
3. Clean, impute, normalize, and engineer analytical features.
4. Export cleansed data to CSV, Excel, and SQLite.


In [ ]:
import pandas as pd
import numpy as np
import os
import sqlite3

raw_path = os.path.join("..", "data", "raw", "global_superstore_raw.csv")
df_raw = pd.read_csv(raw_path)
print(f"Loaded Raw Dataset: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.head(3)


### 1. Data Quality Audit: Missing Values & Duplicates


In [ ]:
print("Duplicate row count:", df_raw.duplicated().sum())
print("
Missing values by column:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])


### 2. Deduplication & Data Cleaning Pipeline


In [ ]:
# 1. Deduplication
df_clean = df_raw.drop_duplicates().copy()

# 2. Clean numeric strings
if df_clean["Sales"].dtype == object:
    df_clean["Sales"] = df_clean["Sales"].astype(str).str.replace("$", "").str.replace(",", "").astype(float)

# 3. Standardize text casing
for col in ["Segment", "Category", "Sub_Category", "Ship_Mode", "Region", "Country"]:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.title()

# 4. Standardize dates
df_clean["Order_Date"] = pd.to_datetime(df_clean["Order_Date"], format="mixed")
df_clean["Ship_Date"] = pd.to_datetime(df_clean["Ship_Date"], format="mixed")
df_clean["Lead_Time_Days"] = (df_clean["Ship_Date"] - df_clean["Order_Date"]).dt.days

# 5. Missing value imputations
df_clean["Discount"] = df_clean["Discount"].fillna(0.0)
df_clean["Shipping_Cost"] = df_clean["Shipping_Cost"].fillna(df_clean["Shipping_Cost"].median())
df_clean["Customer_Name"] = df_clean["Customer_Name"].fillna("Verified Client")
df_clean["Postal_Code"] = df_clean["Postal_Code"].fillna("Unknown")

# 6. Feature Engineering
df_clean["Total_Cost"] = df_clean["Quantity"] * df_clean["Cost_Price"]
df_clean["Profit"] = (df_clean["Sales"] - df_clean["Total_Cost"] - (df_clean["Shipping_Cost"] * 0.4)).round(2)
df_clean["Profit_Margin_Pct"] = ((df_clean["Profit"] / df_clean["Sales"]) * 100).round(2)

print(f"Data Cleaning Completed! Cleaned rows: {len(df_clean)}, Remaining nulls: {df_clean.isnull().sum().sum()}")
df_clean[["Order_ID", "Customer_Name", "Sales", "Profit", "Profit_Margin_Pct", "Lead_Time_Days"]].head()
